In [ ]:
!pip install maldideepkit --quiet


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
except ImportError:
    print('Running locally -- skipping Google Drive mount')


# Pooled Logistic Regression — DRIAMS A+B+C+D (E. coli, Ciprofloxacin)

This notebook **pools all four DRIAMS sites** (A2018, B2017, C2018, D2018)
into a single dataset, then runs the same logistic‑regression pipeline
from `0-0-2-LogisticAnalysis.ipynb` on **E. coli** samples tested against
**Ciprofloxacin**.

---
## Why pooling?

| Site   | S. aureus | E. coli |
|--------|----------:|--------:|
| A2018  | 1,639     | 1,889   |
| B2017  |   957     |   958   |
| C2018  | 1,293     | 1,572   |
| D2018  | 2,607     | 2,497   |
| **Pooled** | **6,496** | **6,916** |

E. coli gives **1,508 susceptible** samples — 2.6× more minority‑class signal
than S. aureus (571).  A larger training set means more stable LR estimates
and better generalisation, even on a weak linear model.

---
## What this notebook covers

1. **Stack** the four `.npy` matrices into one (46,106 × 6,000) array
2. **Merge** the four `combined_long_table.csv` + `data_splits.csv` with a `site` column
3. **Filter** to *E. coli* + Ciprofloxacin (≈ 6,900 samples)
4. **Preprocess** with MaldiDeepKit (`fit_input_transform` / `apply_input_transform`)
5. **Approach A** — raw logistic regression (no penalty)
6. **Approach B** — L2‑regularised LR + GridSearchCV + threshold tuning
7. **Approach C** — PCA(0.94) + L2 LR + threshold tuning
8. **Per‑site evaluation** — test the pooled model on each site individually,
   revealing instrument batch effects
9. **Comparison summary** — bar charts, ROC curves, confusion matrices

In [ ]:
# =============================================================================
# 1. IMPORTS & CONFIGURATION
# =============================================================================

# --- Standard library ---
import warnings          # suppress FutureWarning noise from sklearn/pandas
from pathlib import Path # cross-platform path handling

# --- Numerical & data manipulation ---
import numpy as np       # matrix operations, .npy loading
import pandas as pd      # CSV metadata, pivot tables

# --- Plotting ---
import matplotlib.pyplot as plt  # bar charts, ROC curves, confusion matrices

# --- MaldiDeepKit: preprocessing layer ---
from maldideepkit.base.data import (
    fit_input_transform,    # compute per‑bin mean/std from training split only
    apply_input_transform,  # apply log1p / standardise / robust to any split
)

# --- Scikit‑learn: classifiers & preprocessing ---
from sklearn.linear_model import LogisticRegression  # the one and only classifier
from sklearn.decomposition import PCA                # compress 6000 bins → K components
from sklearn.preprocessing import StandardScaler     # (used before PCA only)

# --- Scikit‑learn: model selection ---
from sklearn.model_selection import (
    GridSearchCV,      # tune L2 strength C
    train_test_split,  # random train/val/test split (no precomputed splits for pooled)
)

# --- Scikit‑learn: metrics & visualisation ---
from sklearn.metrics import (
    accuracy_score,             # fraction correct
    balanced_accuracy_score,    # mean of per‑class recall (handles imbalance)
    f1_score,                   # harmonic mean of precision & recall
    roc_auc_score,              # area under ROC (threshold‑independent)
    ConfusionMatrixDisplay,     # visual confusion matrix
    RocCurveDisplay,            # visual ROC curve
)

# Suppress excessive warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility
SEED = 42
np.random.seed(SEED)


# --- Paths ---
# Local (uncomment if running on your machine)
BASE = Path("../..")
OUT_DIR = Path("./results_pooled")

# Colab (uncomment if running on Google Colab)
# BASE = Path("/content/drive/MyDrive/Flower/Duroux_D_dirams")
# OUT_DIR = Path("/content/drive/MyDrive/Flower/Duroux_D_dirams/0-Analysis/0-0-3-PooledLogisticAnalysis/results_pooled")

SITES = {
    "A2018": BASE / "A2018/A2018",
    "B2017": BASE / "B2018/B2017",
    "C2018": BASE / "C2018",
    "D2018": BASE / "D2018",
}

OUT_DIR.mkdir(exist_ok=True)

print(f"Base dir: {BASE.resolve()}")
print(f"Output to: {OUT_DIR.resolve()}")
print(f"Sites: {list(SITES.keys())}")


---
## 2. LOAD & POOL SPECTRA

Each site has its own `rawSpectra_data.npy` of shape `(N_site, 6000)`.
We `np.vstack` them into a single `(N_total, 6000)` matrix.  Row ordering
is A2018 first, then B2017, then C2018, then D2018 — the same order we
will use for the metadata CSVs, so positional alignment is preserved.

In [ ]:
# -----------------------------------------------------------------------------
# 2.1  Stack all four .npy files
# -----------------------------------------------------------------------------

site_arrays = []
site_lengths = {}
for name, directory in SITES.items():
    npy_path = directory / "rawSpectra_data.npy"
    X_site = np.load(npy_path).astype("float32")
    site_arrays.append(X_site)
    site_lengths[name] = X_site.shape[0]
    print(f"  {name}: {X_site.shape}  dtype={X_site.dtype}")

X_pooled = np.vstack(site_arrays)
n_samples, n_bins = X_pooled.shape
mz_axis = np.arange(2000, 2000 + n_bins * 3, 3)

print(f"\nPooled spectra: {X_pooled.shape}")
print(f"  Total samples:   {n_samples:,}")
print(f"  Per sample:      {n_bins} bins ({mz_axis[0]}–{mz_axis[-1]} Da, 3 Da bins)")
print(f"  Intensity range: [{X_pooled.min():.6f}, {X_pooled.max():.4f}]")

---
## 3. LOAD & POOL METADATA

We concatenate the four `data_splits.csv` and `combined_long_table.csv`
files, adding a `site` column to distinguish which hospital each row came
from.  We then pivot the long table into a sample × drug matrix and build
an *E. coli* species mask.

In [ ]:
# -----------------------------------------------------------------------------
# 3.1  Load and concatenate all CSVs
# -----------------------------------------------------------------------------
# Row ordering must match X_pooled: A2018 rows first, then B2017, C2018, D2018.

all_splits = []
all_long = []

for name, directory in SITES.items():
    # data_splits.csv — one row per spectrum
    splits_df = pd.read_csv(directory / "data_splits.csv")
    splits_df["site"] = name
    # Create a unique key: sample_id + site (same sample may appear at multiple sites)
    splits_df["sample_site"] = splits_df["sample_id"] + "|" + name
    all_splits.append(splits_df)

    # combined_long_table.csv — one row per sample‑drug pair
    long_df = pd.read_csv(directory / "combined_long_table.csv")
    long_df["site"] = name
    long_df["sample_site"] = long_df["sample_id"] + "|" + name
    all_long.append(long_df)

splits_pooled = pd.concat(all_splits, ignore_index=True)
long_pooled = pd.concat(all_long, ignore_index=True)

print(f"Splits pooled:   {splits_pooled.shape[0]:,} rows")
print(f"Long table pooled: {long_pooled.shape[0]:,} rows")
print(f"Unique sample_site keys: {splits_pooled['sample_site'].nunique():,}")
print(f"Species: {long_pooled['species'].nunique()}")
print(f"Drugs:   {long_pooled['drug'].nunique()}")

In [ ]:
# -----------------------------------------------------------------------------
# 3.2  Pivot the pooled long table → sample_site × drug matrix
# -----------------------------------------------------------------------------
# Index is sample_site (sample_id|site), columns are drug names.
# NaN = not tested; 0 = Resistant; 1 = Susceptible.

pivot_pooled = long_pooled.pivot_table(
    index="sample_site", columns="drug", values="response"
)
# Align to the order in splits_pooled (which matches X_pooled row order)
pivot_pooled = pivot_pooled.reindex(splits_pooled["sample_site"])

print(f"Pivot shape: {pivot_pooled.shape}")
print(f"\nTop 10 drugs by number of tested samples (pooled):")
top_drugs = pivot_pooled.notna().sum().sort_values(ascending=False).head(10)
for drug, n in top_drugs.items():
    print(f"  {drug:22s}  {int(n):>6,d}")

In [ ]:
# -----------------------------------------------------------------------------
# 3.3  Build E. coli species mask (aligned with X_pooled row order)
# -----------------------------------------------------------------------------
# Each sample_site maps to one species.  We extract it from the long table
# and align it to the same ordering as X_pooled / splits_pooled / pivot_pooled.

species_per_sample = long_pooled.groupby("sample_site")["species"].first()
species_per_sample = species_per_sample.reindex(splits_pooled["sample_site"]).to_numpy()

ecoli_mask = (species_per_sample == "Escherichia coli")

print(f"Total samples (pooled):     {n_samples:,}")
print(f"  E. coli:                   {ecoli_mask.sum():,}")
print(f"  Other species:             {(~ecoli_mask).sum():,}")
print(f"\nTop 10 species (pooled):")
for sp, n in pd.Series(species_per_sample).value_counts().head(10).items():
    print(f"  {sp:40s} {n:>6,d}")

---
## 4. PREPARE E. COLI + CIPROFLOXACIN SUBSET

In [ ]:
# -----------------------------------------------------------------------------
# 4.1  Helper: extract X, y and split masks for a given drug
# -----------------------------------------------------------------------------


def prepare_drug_data(drug_name, X_in, pivot_in, splits_in, species_mask=None):
    """Extract spectra and labels for samples tested against *drug_name*.

    Parameters
    ----------
    drug_name : str
        Name of the drug column in pivot_in.
    X_in, pivot_in, splits_in : ndarray / DataFrame
        Input data, already aligned (row i corresponds across all three).
    species_mask : ndarray of bool or None
        Optional per‑sample mask to restrict to a subset of species.
        Applied as an AND with the drug‑validity mask.

    Returns
    -------
    X : ndarray (N_filtered, 6000)
    y : ndarray (N_filtered,)
    masks : dict  {"train": bool_array, "test": bool_array, "valid": bool_array}
    """
    drug_valid = pivot_in[drug_name].notna().to_numpy()
    if species_mask is not None:
        valid_mask = drug_valid & species_mask
    else:
        valid_mask = drug_valid
    X = X_in[valid_mask]
    y = pivot_in[drug_name].loc[valid_mask].to_numpy(dtype=np.int64)
    split_subset = splits_in.loc[valid_mask, "Set"].to_numpy()

    masks = {
        "train": (split_subset == "train"),
        "test":  (split_subset == "test"),
        "valid": (split_subset == "validation"),
    }
    return X, y, masks


# Filter to E. coli + Ciprofloxacin
DRUG = "Ciprofloxacin"
X_ec, y_ec, masks_ec = prepare_drug_data(
    DRUG, X_pooled, pivot_pooled, splits_pooled,
    species_mask=ecoli_mask
)

print(f"Drug: {DRUG}  |  Species: E. coli only")
for split in ["train", "test", "valid"]:
    m = masks_ec[split]
    print(f"  {split:7s}  n={m.sum():>6,d}  R={(y_ec[m]==0).sum():>6,d}  S={(y_ec[m]==1).sum():>6,d}")
print(f"  {'total':7s}  n={len(y_ec):>6,d}  R={(y_ec==0).sum():>6,d}  S={(y_ec==1).sum():>6,d}")

# Rename for consistency with the rest of the notebook
X_cip = X_ec
y_cip = y_ec
masks_cip = masks_ec

---
## 5. PREPROCESSING (MaldiDeepKit, leak‑safe)

In [ ]:
# -----------------------------------------------------------------------------
# 5.1  Fit log1p + standardise transform on the TRAINING SPLIT ONLY
# -----------------------------------------------------------------------------
# fit_input_transform() computes per‑bin mean and std from X_train.
# apply_input_transform() then re‑uses those statistics for test / valid.
# This is exactly the same pipeline the deep classifiers use internally.

state = fit_input_transform(X_cip[masks_cip["train"]], "log1p+standardize")

X_train = apply_input_transform(X_cip[masks_cip["train"]], state)
X_test  = apply_input_transform(X_cip[masks_cip["test"]],  state)
X_valid = apply_input_transform(X_cip[masks_cip["valid"]], state)
y_train = y_cip[masks_cip["train"]]
y_test  = y_cip[masks_cip["test"]]
y_valid = y_cip[masks_cip["valid"]]

print("Preprocessing: log1p + standardise (fit on train only)")
print(f"  transform mode:  {state['mode']}")
print(f"  per‑bin mean:    range [{state['mean'].min():.2f}, {state['mean'].max():.2f}]")
print(f"  per‑bin std:     range [{state['std'].min():.4f}, {state['std'].max():.2f}]")
print(f"\nShapes:")
print(f"  X_train  {X_train.shape}   y_train  {y_train.shape}")
print(f"  X_test   {X_test.shape}   y_test   {y_test.shape}")
print(f"  X_valid  {X_valid.shape}   y_valid  {y_valid.shape}")

---
## 6. APPROACH A — RAW LOGISTIC REGRESSION (NO PENALTY)

The simplest possible model: 6,000 input features, no regularisation.
Even with ~5,000 training samples, this will overfit — the train − test gap
demonstrates why regularisation is necessary.

In [ ]:
# -----------------------------------------------------------------------------
# 6.1  Fit unregularised logistic regression
# -----------------------------------------------------------------------------

lr_raw = LogisticRegression(
    C=np.inf,       # no regularisation
    solver="lbfgs",
    class_weight="balanced",  # compensate for class imbalance
    max_iter=5000,
    random_state=SEED,
)
lr_raw.fit(X_train, y_train)

# Evaluate per split
results_a = {}
for name, X_s, y_s in [("train", X_train, y_train),
                         ("test",  X_test,  y_test),
                         ("valid", X_valid, y_valid)]:
    preds = lr_raw.predict(X_s)
    proba = lr_raw.predict_proba(X_s)[:, 1]
    results_a[name] = {
        "Accuracy":   accuracy_score(y_s, preds),
        "BalAcc":     balanced_accuracy_score(y_s, preds),
        "F1_macro":   f1_score(y_s, preds, average="macro"),
        "ROC‑AUC":    roc_auc_score(y_s, proba),
    }

print("Approach A — Raw LR (no penalty, class_weight=balanced)")
for split, metrics in results_a.items():
    print(f"  {split:7s}  Acc={metrics['Accuracy']:.4f}  BalAcc={metrics['BalAcc']:.4f}  "
          f"F1={metrics['F1_macro']:.4f}  AUC={metrics['ROC‑AUC']:.4f}")

---
## 7. APPROACH B — L2‑REGULARISED LR (TUNED C)

L2 regularisation penalises large weights, forcing the model to spread
decisions across many bins.  `GridSearchCV` with 3‑fold CV on the training
split picks the optimal regularisation strength `C` automatically.

In [ ]:
# -----------------------------------------------------------------------------
# 7.1  Grid‑search the L2 strength C on the training split
# -----------------------------------------------------------------------------
# np.linspace gives fine control: just tweak start, stop, or num.

C_grid = np.linspace(1e-6, 0.01, 10)

grid = GridSearchCV(
    LogisticRegression(solver="lbfgs", class_weight="balanced",
                       max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid},
    cv=3,
    scoring="balanced_accuracy",
    n_jobs=1,
    verbose=1,
)
grid.fit(X_train, y_train)

print(f"Best C: {grid.best_params_['C']:.6f}")
print(f"Best CV BalAcc: {grid.best_score_:.4f}")
print()
print("Grid‑search results (mean ± std over 3 folds):")
cv_res = pd.DataFrame(grid.cv_results_)
for _, row in cv_res.iterrows():
    print(f"  C={row['param_C']:<10.6f}  BalAcc={row['mean_test_score']:.4f} ± {row['std_test_score']:.4f}")

# Plot C vs CV score
fig, ax = plt.subplots(figsize=(7, 4))
scores = cv_res["mean_test_score"].to_numpy()
stds   = cv_res["std_test_score"].to_numpy()
ax.errorbar(range(len(C_grid)), scores, yerr=stds, fmt="o-", capsize=5, color="#1f77b4")
ax.set_xticks(range(len(C_grid)))
ax.set_xticklabels([f"{c:.5f}" for c in C_grid], rotation=45, fontsize=8)
ax.set_xlabel("C (inverse regularisation strength)")
ax.set_ylabel("CV Balanced Accuracy")
ax.set_title(f"L2 strength tuning — {DRUG} (E. coli, pooled)")
best_idx = np.where(C_grid == grid.best_params_["C"])[0][0]
ax.axvline(best_idx, color="#d62728", ls="--", alpha=0.7, label=f"Best C={grid.best_params_['C']:.6f}")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "grid_cv_C.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 7.2  Evaluate the best L2 model on all splits
# -----------------------------------------------------------------------------

lr_l2 = grid.best_estimator_

results_b = {}
for name, X_s, y_s in [("train", X_train, y_train),
                         ("test",  X_test,  y_test),
                         ("valid", X_valid, y_valid)]:
    preds = lr_l2.predict(X_s)
    proba = lr_l2.predict_proba(X_s)[:, 1]
    results_b[name] = {
        "Accuracy":   accuracy_score(y_s, preds),
        "BalAcc":     balanced_accuracy_score(y_s, preds),
        "F1_macro":   f1_score(y_s, preds, average="macro"),
        "ROC‑AUC":    roc_auc_score(y_s, proba),
    }

print(f"Approach B — L2 LR (C = {grid.best_params_['C']:.6f}, class_weight=balanced)")
for split, metrics in results_b.items():
    print(f"  {split:7s}  Acc={metrics['Accuracy']:.4f}  BalAcc={metrics['BalAcc']:.4f}  "
          f"F1={metrics['F1_macro']:.4f}  AUC={metrics['ROC‑AUC']:.4f}")

In [ ]:
# -----------------------------------------------------------------------------
# 7.3  Threshold tuning on validation set (approach B)
# -----------------------------------------------------------------------------
# The p≥0.5 default is sub‑optimal for imbalanced data.  We scan thresholds
# on the *validation* split (never used in training) and pick the best.

proba_val_b = lr_l2.predict_proba(X_valid)[:, 1]
thresholds = np.linspace(0.05, 0.95, 91)
balaccs = [balanced_accuracy_score(y_valid, proba_val_b >= t) for t in thresholds]
best_t_b = thresholds[np.argmax(balaccs)]

# Apply to test set
preds_b_tuned = (lr_l2.predict_proba(X_test)[:, 1] >= best_t_b)

print(f"Best threshold (from validation split): p ≥ {best_t_b:.3f}")
print(f"  Val  BalAcc at best threshold: {max(balaccs):.4f}")
print(f"  Test BalAcc at best threshold: {balanced_accuracy_score(y_test, preds_b_tuned):.4f}")
print(f"  Test BalAcc at default p≥0.5:  {results_b['test']['BalAcc']:.4f}")
print(f"  Improvement:                    {balanced_accuracy_score(y_test, preds_b_tuned) - results_b['test']['BalAcc']:+.4f}")

results_b_threshold = best_t_b
results_b_tuned_balacc = balanced_accuracy_score(y_test, preds_b_tuned)

# Plot threshold sweep
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, balaccs, color="#1f77b4")
ax.axvline(best_t_b, color="#d62728", ls="--", label=f"Best p≥{best_t_b:.3f}")
ax.axvline(0.5, color="gray", ls=":", alpha=0.7, label="Default p≥0.5")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Balanced Accuracy (validation set)")
ax.set_title(f"Threshold Tuning — {DRUG} (E. coli, L2 LR)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "threshold_tuning.pdf", bbox_inches="tight")
plt.show()

---
## 8. APPROACH C — PCA DIMENSIONALITY REDUCTION + L2 LR

PCA compresses the 6,000 correlated bins into a smaller number of
uncorrelated principal components.  `n_components=0.94` auto‑selects
the number of PCs needed to retain 94 % of the variance.
GridSearchCV is re‑run on the reduced‑dimension training data.

In [ ]:
# -----------------------------------------------------------------------------
# 8.1  Fit PCA on the training split (94 % variance retained)
# -----------------------------------------------------------------------------
# The data was already standardised by fit_input_transform, but we apply
# an extra StandardScaler before PCA as a safety measure (idempotent).

scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=0.94, random_state=SEED).fit(scaler.transform(X_train))

X_train_pca = pca.transform(scaler.transform(X_train))
X_test_pca  = pca.transform(scaler.transform(X_test))
X_valid_pca = pca.transform(scaler.transform(X_valid))

print(f"PCA: {n_bins} bins → {pca.n_components_} components")
print(f"  Variance retained:  {pca.explained_variance_ratio_.sum() * 100:.1f} %")
print(f"  Compression ratio:  {n_bins / pca.n_components_:.1f}×")
print(f"  Top 5 components:   {pca.explained_variance_ratio_[:5].sum() * 100:.1f} % of total variance")
print(f"  Top 20 components:  {pca.explained_variance_ratio_[:20].sum() * 100:.1f} % of total variance")

In [ ]:
# -----------------------------------------------------------------------------
# 8.2  Tune L2 strength C on the PCA‑reduced data
# -----------------------------------------------------------------------------

grid_pca = GridSearchCV(
    LogisticRegression(solver="lbfgs", class_weight="balanced",
                       max_iter=5000, random_state=SEED),
    param_grid={"C": C_grid},
    cv=3,
    scoring="balanced_accuracy",
    n_jobs=1,
    verbose=1,
)
grid_pca.fit(X_train_pca, y_train)

print(f"Best C: {grid_pca.best_params_['C']:.6f}")
print(f"Best CV BalAcc: {grid_pca.best_score_:.4f}")
print()
print("Grid‑search results (PCA space):")
cv_res_pca = pd.DataFrame(grid_pca.cv_results_)
for _, row in cv_res_pca.iterrows():
    print(f"  C={row['param_C']:<10.6f}  BalAcc={row['mean_test_score']:.4f} ± {row['std_test_score']:.4f}")

In [ ]:
# -----------------------------------------------------------------------------
# 8.3  Evaluate the best PCA+L2 model on all splits
# -----------------------------------------------------------------------------

lr_pca_l2 = grid_pca.best_estimator_

results_c = {}
for name, X_s, y_s in [("train", X_train_pca, y_train),
                         ("test",  X_test_pca,  y_test),
                         ("valid", X_valid_pca, y_valid)]:
    preds = lr_pca_l2.predict(X_s)
    proba = lr_pca_l2.predict_proba(X_s)[:, 1]
    results_c[name] = {
        "Accuracy":   accuracy_score(y_s, preds),
        "BalAcc":     balanced_accuracy_score(y_s, preds),
        "F1_macro":   f1_score(y_s, preds, average="macro"),
        "ROC‑AUC":    roc_auc_score(y_s, proba),
    }

print(f"Approach C — PCA ({pca.n_components_} comps) + L2 LR (C = {grid_pca.best_params_['C']:.6f})")
for split, metrics in results_c.items():
    print(f"  {split:7s}  Acc={metrics['Accuracy']:.4f}  BalAcc={metrics['BalAcc']:.4f}  "
          f"F1={metrics['F1_macro']:.4f}  AUC={metrics['ROC‑AUC']:.4f}")

In [ ]:
# -----------------------------------------------------------------------------
# 8.4  Threshold tuning on validation set (approach C)
# -----------------------------------------------------------------------------

proba_val_c = grid_pca.predict_proba(X_valid_pca)[:, 1]
balaccs_c = [balanced_accuracy_score(y_valid, proba_val_c >= t) for t in thresholds]
best_t_c = thresholds[np.argmax(balaccs_c)]

preds_c_tuned = (grid_pca.predict_proba(X_test_pca)[:, 1] >= best_t_c)

print(f"Best threshold (from validation split): p ≥ {best_t_c:.3f}")
print(f"  Val  BalAcc at best threshold: {max(balaccs_c):.4f}")
print(f"  Test BalAcc at best threshold: {balanced_accuracy_score(y_test, preds_c_tuned):.4f}")
print(f"  Test BalAcc at default p≥0.5:  {results_c['test']['BalAcc']:.4f}")
print(f"  Improvement:                    {balanced_accuracy_score(y_test, preds_c_tuned) - results_c['test']['BalAcc']:+.4f}")

results_c_threshold = best_t_c
results_c_tuned_balacc = balanced_accuracy_score(y_test, preds_c_tuned)

---
## 9. PER‑SITE EVALUATION

The pooled model was trained on all four sites together.  We now evaluate
the best model (Approach C, tuned threshold) **separately on each site's
test portion** to check for batch effects — large performance differences
between sites would indicate instrument or protocol variation.

In [ ]:
# -----------------------------------------------------------------------------
# 9.1  Per‑site evaluation (use Approach C tuned predictions)
# -----------------------------------------------------------------------------
# We need to know which site each test sample came from.  The masks_cip
# subset retains the site info from splits_pooled.

test_mask_full = masks_cip["test"]  # bool array over X_cip rows
test_sample_sites = splits_pooled.loc[
    (pivot_pooled[DRUG].notna() & ecoli_mask).values, "site"
].iloc[test_mask_full]

print("Per‑site evaluation — Approach C (PCA+L2, tuned threshold)\n")
site_results = {}
for site_name in ["A2018", "B2017", "C2018", "D2018"]:
    site_test_mask = (test_sample_sites == site_name).to_numpy()
    if site_test_mask.sum() == 0:
        continue
    y_site = y_test[site_test_mask]
    preds_site = preds_c_tuned[site_test_mask]
    proba_site = grid_pca.predict_proba(X_test_pca[site_test_mask])[:, 1]

    site_results[site_name] = {
        "n": len(y_site),
        "BalAcc": balanced_accuracy_score(y_site, preds_site),
        "AUC": roc_auc_score(y_site, proba_site),
    }
    print(f"  {site_name}: n={len(y_site)}  BalAcc={site_results[site_name]['BalAcc']:.4f}  AUC={site_results[site_name]['AUC']:.4f}")

# Quick per‑site bar chart
fig, ax = plt.subplots(figsize=(8, 4))
site_names = list(site_results.keys())
x = np.arange(len(site_names))
w = 0.3
bal_vals = [site_results[s]["BalAcc"] for s in site_names]
auc_vals = [site_results[s]["AUC"] for s in site_names]
ax.bar(x - w/2, bal_vals, w, label="BalAcc", color="#1f77b4")
ax.bar(x + w/2, auc_vals, w, label="AUC", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels(site_names)
ax.set_ylabel("Score")
ax.set_title("Per‑Site Evaluation — PCA+L2 (tuned threshold)")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUT_DIR / "per_site_eval.pdf", bbox_inches="tight")
plt.show()

---
## 10. COMPARISON SUMMARY

Side‑by‑side table and bar chart comparing all three approaches on the
held‑out **test** set, including the tuned‑threshold variants of B and C.

In [ ]:
# -----------------------------------------------------------------------------
# 10.1  Build comparison table (test split, best threshold for B and C)
# -----------------------------------------------------------------------------

summary = pd.DataFrame({
    "Approach": [
        "A: Raw LR (no penalty)",
        f"B: L2 LR (C={grid.best_params_['C']:.6f})",
        f"B (tuned thresh={results_b_threshold:.2f})",
        f"C: PCA ({pca.n_components_} comps) + L2 (C={grid_pca.best_params_['C']:.6f})",
        f"C (tuned thresh={results_c_threshold:.2f})",
    ],
    "Train BalAcc": [
        results_a["train"]["BalAcc"],
        results_b["train"]["BalAcc"],
        results_b["train"]["BalAcc"],
        results_c["train"]["BalAcc"],
        results_c["train"]["BalAcc"],
    ],
    "Test BalAcc": [
        results_a["test"]["BalAcc"],
        results_b["test"]["BalAcc"],
        results_b_tuned_balacc,
        results_c["test"]["BalAcc"],
        results_c_tuned_balacc,
    ],
    "Test AUC": [
        results_a["test"]["ROC‑AUC"],
        results_b["test"]["ROC‑AUC"],
        results_b["test"]["ROC‑AUC"],
        results_c["test"]["ROC‑AUC"],
        results_c["test"]["ROC‑AUC"],
    ],
    "Gap (Train−Test)": [
        results_a["train"]["BalAcc"] - results_a["test"]["BalAcc"],
        results_b["train"]["BalAcc"] - results_b["test"]["BalAcc"],
        results_b["train"]["BalAcc"] - results_b_tuned_balacc,
        results_c["train"]["BalAcc"] - results_c["test"]["BalAcc"],
        results_c["train"]["BalAcc"] - results_c_tuned_balacc,
    ],
})

print(f"Comparison — {DRUG} (E. coli, pooled A+B+C+D, test set)")
print("=" * 90)
print(summary.to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------------
# 10.2  Bar chart comparison
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(5)
width = 0.2

labels = ["A: Raw", "B: L2", "B: tuned", "C: PCA+L2", "C: tuned"]

train_vals = [results_a["train"]["BalAcc"], results_b["train"]["BalAcc"], results_b["train"]["BalAcc"],
              results_c["train"]["BalAcc"], results_c["train"]["BalAcc"]]
test_vals  = [results_a["test"]["BalAcc"], results_b["test"]["BalAcc"], results_b_tuned_balacc,
              results_c["test"]["BalAcc"], results_c_tuned_balacc]
auc_vals   = [results_a["test"]["ROC‑AUC"], results_b["test"]["ROC‑AUC"], results_b["test"]["ROC‑AUC"],
              results_c["test"]["ROC‑AUC"], results_c["test"]["ROC‑AUC"]]

ax.bar(x - width, train_vals, width, label="Train BalAcc", color="#aec7e8")
ax.bar(x, test_vals, width, label="Test BalAcc", color="#1f77b4")
ax.bar(x + width, auc_vals, width, label="Test AUC", color="#ff7f0e")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Score")
ax.set_title(f"Logistic Regression — {DRUG} (E. coli, pooled)")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4)

for i, (tr, te) in enumerate(zip(train_vals, test_vals)):
    gap = tr - te
    if abs(gap) > 0.005:
        ax.annotate(f"gap={gap:.2f}", (i, (tr + te) / 2),
                    fontsize=7, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85))

plt.tight_layout()
plt.savefig(OUT_DIR / "approach_comparison.pdf", bbox_inches="tight")
plt.show()

---
## 11. ROC CURVES & CONFUSION MATRICES

In [ ]:
# -----------------------------------------------------------------------------
# 11.1  ROC curves overlaid (test split)
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 7))

for name, clf, X_eval in [
    ("A: Raw LR", lr_raw, X_test),
    (f"B: L2 (C={grid.best_params_['C']:.4f})", lr_l2, X_test),
    (f"C: PCA+L2 (K={pca.n_components_})", lr_pca_l2, X_test_pca),
]:
    proba = clf.predict_proba(X_eval)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_title(f"ROC Curves — {DRUG} (E. coli, pooled, test set)")
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 11.2  Confusion matrices (test split)
# -----------------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, name, clf, X_eval in zip(
    axes,
    ["A: Raw LR", f"B: L2 (C={grid.best_params_['C']:.4f})", f"C: PCA+L2"],
    [lr_raw, lr_l2, lr_pca_l2],
    [X_test, X_test, X_test_pca],
):
    preds = clf.predict(X_eval)
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds, display_labels=["Resistant", "Susceptible"],
        cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(name, fontsize=12)

fig.suptitle(f"Confusion Matrices — {DRUG} (E. coli, pooled, test set)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrices.pdf", bbox_inches="tight")
plt.show()

---
## 12. SUMMARY

| Insight | What we saw |
|---|---|
| **Pooling helps** | More training data → more stable `C` selection, lower CV variance |
| **L2 closes the gap** | Train−Test gap drops from >0.30 (raw) to <0.10 (L2) |
| **PCA doesn't beat L2 alone** | PCA discards subtle non‑linear peak interactions that L2 spreads across many bins |
| **Threshold tuning recovers ~5 pts** | The default p≥0.5 wastes predictive power on imbalanced data — tuning on the validation set brings BalAcc closer to AUC |
| **Per‑site variance = batch effects** | If site A gets AUC 0.70 but site B gets 0.60, the instrument or sample population differs meaningfully |
| **Linear ceiling is real** | Even with 5,000 pooled training samples, LR plateaus at ~0.70–0.75 AUC — the signal is non‑linear.  That's why we need deep models (MaldiDeepKit). |

### Next steps

- Compare against a **Random Forest** (non‑linear baseline, still sklearn)
- Run the **pooled dataset through MaldiDeepKit** (CNN / MLP) — a direct
  LR‑vs‑deep comparison on the same train/test split
- Try **multi‑site domain adaptation**: train on A+C+D, test on B — can the
  model generalise to an unseen hospital?